# RelBench rel-amazon Dataset: Download, Extract, and Prepare for PRMP Diagnostic

This notebook demonstrates the **Amazon Video Games product reviews relational dataset** prepared for PRMP (Predictive Residual Message Passing) cross-table predictability diagnostic.

**What this dataset contains:**
- 50K reviews across 3.4K products and 10.9K customers
- 10K sampled examples with 21 encoded features (time, text hash, helpful votes)
- 2 FK links (product→review, customer→review) with cardinality statistics
- 2K aligned parent-child feature matrices with aggregated parent features (mean rating, std rating, review count, helpful stats)
- Schema-validated `exp_sel_data_out` format for R² computation

**What this notebook does:**
1. Loads the pre-processed relational dataset
2. Explores the relational schema (tables, FK links, cardinality statistics)
3. Extracts aligned parent-child feature matrices
4. Computes R² (cross-table predictability) via linear regression
5. Visualizes the results

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# All packages used are pre-installed on Colab; install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

In [ ]:
import json
import os

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-b2d5b0-predictive-residual-message-passing-filt/main/dataset_iter1_relbench_rel_am/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded dataset with keys: {list(data.keys())}")
print(f"Number of dataset groups: {len(data['datasets'])}")
print(f"Dataset name: {data['datasets'][0]['dataset']}")
print(f"Number of examples: {len(data['datasets'][0]['examples'])}")

In [ ]:
# === Config ===
# Tunable parameters for the demo (set to minimum for fast execution)
# Original values from the full pipeline are commented out

# MAX_EXAMPLES = 10_000  # original
MAX_EXAMPLES = 3  # demo: use all available mini examples

# ALIGN_SAMPLE = 2_000  # original
ALIGN_SAMPLE = 50  # demo: aligned parent-child samples for R² computation

## 1. Explore Relational Schema

The dataset has a **relational triangle** structure: `product → review ← customer`. Each review is a child record linked to both a parent product and a parent customer via foreign keys.

In [ ]:
ds_info = data["metadata"]["datasets_info"]["amazon_video_games"]

print("=== Relational Tables ===")
for table_name, table_info in ds_info["tables"].items():
    print(f"\n  Table: {table_name}")
    print(f"    Rows: {table_info['num_rows']:,}")
    print(f"    Columns: {table_info['columns']}")
    print(f"    Primary key: {table_info['pkey_col']}")
    if "aggregated_feature_names" in table_info:
        print(f"    Aggregated features: {table_info['aggregated_feature_names']}")
    if "feature_names" in table_info:
        print(f"    Encoded features ({len(table_info['feature_names'])}): {table_info['feature_names'][:5]}...")
    if "fkey_col_to_pkey_table" in table_info:
        print(f"    Foreign keys: {table_info['fkey_col_to_pkey_table']}")

print(f"\nTime range: {ds_info['time_range']['start']} to {ds_info['time_range']['end']}")
print(f"Total reviews: {ds_info['total_reviews']:,}")
print(f"Sampled examples: {ds_info['num_examples_sampled']:,}")
print(f"Aligned sample size: {ds_info['align_sample_size']:,}")

## 2. FK Link Cardinality Statistics

Each FK link has cardinality statistics describing how many child records (reviews) each parent (product or customer) has. This is critical for PRMP — higher cardinality means more information flows through the message-passing link.

In [ ]:
fk_links = ds_info["fk_links"]

print("=== FK Link Cardinality Statistics ===\n")
for link_name, link_info in fk_links.items():
    print(f"  {link_name}: {link_info['parent_table']} → {link_info['child_table']}")
    print(f"    FK column: {link_info['fk_col']}")
    print(f"    Parents: {link_info['num_parents']:,}  |  Children: {link_info['num_children']:,}")
    print(f"    Cardinality  mean={link_info['cardinality_mean']:.2f}  "
          f"median={link_info['cardinality_median']:.1f}  "
          f"std={link_info['cardinality_std']:.2f}")
    print(f"    Range: [{link_info['cardinality_min']}, {link_info['cardinality_max']}]  "
          f"IQR: [{link_info['cardinality_p25']}, {link_info['cardinality_p75']}]")
    print(f"    Parent features: {link_info['parent_feature_names']}")
    print(f"    Child features ({len(link_info['child_feature_names'])}): "
          f"{link_info['child_feature_names'][:5]}...")
    print()

## 3. Inspect Examples

Each example represents a review with encoded features (input) and a rating (output). The features include time encodings, text hash features, and helpful vote counts.

In [ ]:
examples = data["datasets"][0]["examples"][:MAX_EXAMPLES]
print(f"Number of examples: {len(examples)}\n")

for i, ex in enumerate(examples):
    features = json.loads(ex["input"])
    print(f"--- Example {i} ---")
    print(f"  Rating (output): {ex['output']}")
    print(f"  Product ID: {ex['metadata_product_id']}")
    print(f"  Customer ID: {ex['metadata_customer_id']}")
    print(f"  Fold: {ex['metadata_fold']}  |  Task: {ex['metadata_task_type']}")
    print(f"  Features ({len(features)}):")
    for fname, fval in list(features.items())[:6]:
        print(f"    {fname}: {fval}")
    if len(features) > 6:
        print(f"    ... ({len(features) - 6} more features)")
    print()

## 4. Extract Aligned Parent-Child Feature Matrices

The dataset provides pre-aligned parent and child feature matrices for each FK link. For each sampled review, the corresponding parent (product or customer) features are aligned row-by-row. This enables direct R² computation to measure cross-table predictability.

In [ ]:
aligned_data = {}

for link_name, link_info in fk_links.items():
    parent_feats = np.array(link_info["aligned_parent_features_sample"][:ALIGN_SAMPLE], dtype=np.float32)
    child_feats = np.array(link_info["aligned_child_features_sample"][:ALIGN_SAMPLE], dtype=np.float32)

    aligned_data[link_name] = {
        "parent_feats": parent_feats,
        "child_feats": child_feats,
        "parent_names": link_info["parent_feature_names"],
        "child_names": link_info["child_feature_names"],
    }

    print(f"{link_name}:")
    print(f"  Parent features shape: {parent_feats.shape}  names: {link_info['parent_feature_names']}")
    print(f"  Child features shape:  {child_feats.shape}  ({len(link_info['child_feature_names'])} features)")
    print(f"  Parent stats: mean={parent_feats.mean():.3f}, std={parent_feats.std():.3f}")
    print(f"  Child stats:  mean={child_feats.mean():.3f}, std={child_feats.std():.3f}")
    print()

## 5. Compute R² Cross-Table Predictability

The core PRMP diagnostic: for each FK link, fit a linear regression from parent features to each child feature dimension. The R² score measures how much of the child feature variance is explained by the parent features. Higher R² means stronger cross-table predictability — the parent table carries useful information for predicting child records.

In [ ]:
r2_results = {}

for link_name, ad in aligned_data.items():
    X_parent = ad["parent_feats"]
    Y_child = ad["child_feats"]
    child_names = ad["child_names"]

    print(f"=== {link_name} ===")
    print(f"  Fitting LinearRegression: {X_parent.shape[1]} parent features → {Y_child.shape[1]} child features")
    print(f"  Using {X_parent.shape[0]} aligned samples\n")

    per_feature_r2 = []
    for j in range(Y_child.shape[1]):
        y = Y_child[:, j]
        # Skip constant features (zero variance)
        if np.std(y) < 1e-8:
            per_feature_r2.append(0.0)
            continue
        reg = LinearRegression()
        reg.fit(X_parent, y)
        y_pred = reg.predict(X_parent)
        r2 = r2_score(y, y_pred)
        per_feature_r2.append(max(0.0, r2))  # clip negative R² to 0

    # Overall R² (multivariate)
    reg_multi = LinearRegression()
    reg_multi.fit(X_parent, Y_child)
    Y_pred = reg_multi.predict(X_parent)
    overall_r2 = r2_score(Y_child, Y_pred)

    r2_results[link_name] = {
        "per_feature_r2": per_feature_r2,
        "child_names": child_names,
        "overall_r2": overall_r2,
        "mean_r2": float(np.mean(per_feature_r2)),
    }

    print(f"  Per-feature R² (top 10):")
    sorted_feats = sorted(zip(child_names, per_feature_r2), key=lambda x: -x[1])
    for fname, r2 in sorted_feats[:10]:
        bar = "#" * int(r2 * 40)
        print(f"    {fname:20s}  R²={r2:.4f}  {bar}")
    print(f"\n  Overall multivariate R²: {overall_r2:.4f}")
    print(f"  Mean per-feature R²:    {np.mean(per_feature_r2):.4f}")
    print()

## 6. Visualization

Compare R² cross-table predictability across the two FK links (product→review vs customer→review), and show per-feature R² distributions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: Overall R² comparison ---
ax = axes[0]
link_names = list(r2_results.keys())
overall_r2s = [r2_results[ln]["overall_r2"] for ln in link_names]
mean_r2s = [r2_results[ln]["mean_r2"] for ln in link_names]
short_names = [ln.replace("_to_review", "").replace("_to_rating", "") for ln in link_names]

x = np.arange(len(link_names))
width = 0.35
bars1 = ax.bar(x - width/2, overall_r2s, width, label="Overall R²", color="#2196F3")
bars2 = ax.bar(x + width/2, mean_r2s, width, label="Mean per-feature R²", color="#FF9800")
ax.set_ylabel("R² Score")
ax.set_title("Cross-Table Predictability by FK Link")
ax.set_xticks(x)
ax.set_xticklabels(short_names)
ax.legend()
ax.set_ylim(0, max(max(overall_r2s), max(mean_r2s)) * 1.3 + 0.01)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

# --- Plot 2 & 3: Per-feature R² for each FK link ---
for idx, (link_name, res) in enumerate(r2_results.items()):
    ax = axes[idx + 1]
    r2s = res["per_feature_r2"]
    names = res["child_names"]
    # Sort by R² descending
    sorted_pairs = sorted(zip(names, r2s), key=lambda x: -x[1])
    snames, sr2s = zip(*sorted_pairs)

    colors = ["#4CAF50" if r > 0.05 else "#BDBDBD" for r in sr2s]
    bars = ax.barh(range(len(sr2s)), sr2s, color=colors)
    ax.set_yticks(range(len(snames)))
    ax.set_yticklabels(snames, fontsize=7)
    ax.set_xlabel("R² Score")
    short = link_name.replace("_to_review", "").replace("_to_rating", "")
    ax.set_title(f"Per-Feature R²: {short}→review")
    ax.invert_yaxis()
    ax.axvline(x=0.05, color="red", linestyle="--", alpha=0.5, label="R²=0.05")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("r2_cross_table_predictability.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: r2_cross_table_predictability.png")

## 7. Summary

Final summary table comparing cross-table predictability for each FK link.

In [ ]:
print("=" * 70)
print("PRMP Cross-Table Predictability Diagnostic — Summary")
print("=" * 70)
print(f"{'FK Link':<25s} {'Overall R²':>12s} {'Mean R²':>10s} {'# Features':>12s}")
print("-" * 70)

for link_name, res in r2_results.items():
    n_feats = len(res["per_feature_r2"])
    print(f"{link_name:<25s} {res['overall_r2']:>12.4f} {res['mean_r2']:>10.4f} {n_feats:>12d}")

print("-" * 70)
print(f"\nDataset: Amazon Video Games ({ds_info['total_reviews']:,} reviews)")
print(f"Aligned samples used for R²: {ALIGN_SAMPLE}")
print(f"Child features: {len(fk_links['product_to_review']['child_feature_names'])} dimensions")
print(f"\nInterpretation:")
print("  - Higher R² = parent features explain more child feature variance")
print("  - This quantifies how much information flows across FK links")
print("  - PRMP uses this to weight residual message passing accordingly")